In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel("data.xlsx")
df.head()

Matplotlib is building the font cache; this may take a moment.


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
print(df.shape)
df.info()

(541909, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [3]:
df.dropna(subset=['Description'], inplace=True)
df.drop(
    df[df['InvoiceNo'].astype(str).str.startswith('C')].index,
    inplace=True
)
df.drop(df[df['Quantity'] <= 0].index, inplace=True)

print(df.shape)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\2064947059.py:1: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.dropna(subset=['Description'], inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\2064947059.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\2064947059.py:6: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(df[df['Quantity'] <= 0].index, inplace=True)


(530693, 8)


In [4]:
# Preview awal sebelum kandidat nonproduk dibersihkan; basket final dibuat ulang setelah pemeriksaan.
basket = pd.crosstab(
    index=df['InvoiceNo'],
    columns=df['Description']
).gt(0)

basket.head()

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,returned,taig adjust,test,to push order througha s stock was,website fixed,wrongly coded 20713,wrongly coded 23343,wrongly marked,wrongly marked 23343,wrongly sold (22719) barcode
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536366,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536367,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536368,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536369,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [5]:
kata = r'return|test|wrongly|adjust|coded'

df.loc[
    df['Description'].str.contains(kata, case=False, na=False),
    ['InvoiceNo', 'StockCode', 'Description', 'Quantity']
].head(20)

,InvoiceNo,StockCode,Description,Quantity
112219,545875,21736,Adjustment,34
117895,546408,22467,wrongly sold (22719) barcode,170
166639,550947,21319,adjustment,3
166640,550948,17109D,Adjustment,14
166646,550951,84531A,returned,2
166647,550952,84531B,returned,2
169714,551240,51020A,adjustment,41
170485,551304,48189,adjustment,2
195520,553719,85017C,adjustment,1
263885,560040,23343,came coded as 20713,3100


In [6]:
df.loc[
    df['Description'].str.contains(kata, case=False, na=False),
    'Description'
].value_counts()

Description
adjustment                      14
Adjust bad debt                  3
Adjustment                       2
returned                         2
test                             2
taig adjust                      2
wrongly sold (22719) barcode     1
came coded as 20713              1
wrongly coded 20713              1
Amazon Adjustment                1
wrongly marked 23343             1
wrongly coded 23343              1
wrongly marked                   1
amazon adjust                    1
dotcom adjust                    1
Name: count, dtype: int64

In [7]:
deskripsi_bukan_produk = {
    'postage',
    'dotcom postage',
    'carriage',
    'next day carriage',
    'manual',
    'check',
    'check?',
    'found',
    'found by jackie',
    'found in w/hse',
    'found box',
    'adjustment',
    'adjust bad debt',
    'test',
    'returned',
    'taig adjust',
    'bank charges',
    'amazon fee',
    'amazon',
    'amazon adjustment',
    'amazon adjust',
    'amazon sales',
    'dotcom',
    'dotcom adjust',
    'dotcomstock',
    'had been put aside',
    'had been put aside.',
    'high resolution image',
    'mailout',
    'samples',
    'lighthouse trading zero invc incorr',
    'marked as 23343',
    'sale error',
    'allocate stock for dotcom orders ta',
    'came coded as 20713',
    'damaged',
    'did a credit and did not tick ret',
    'for online retail orders',
    'incorrectly credited c550456 see 47',
    'michel oops',
    'on cargo order',
    'rcvd be air temp fix for dotcom sit',
    'wrongly sold (22719) barcode',
    'wrongly coded 20713',
    'wrongly coded 23343',
    'wrongly marked',
    'wrongly marked 23343',
    'to push order througha s stock was',
    'website fixed',
}

kode_bukan_produk = {
    'POST', 'DOT', 'C2', '23444', 'M',
    'BANK CHARGES', 'AMAZONFEE', 'S', 'B'
}

deskripsi_normal = (
    df['Description']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.casefold()
)
kode_normal = df['StockCode'].astype(str).str.strip().str.upper()
mask_bukan_produk = (
    deskripsi_normal.isin(deskripsi_bukan_produk)
    | kode_normal.isin(kode_bukan_produk)
)

print('Rincian baris nonproduk yang akan dihapus:')
print(df.loc[mask_bukan_produk, 'Description'].value_counts().to_string())
jumlah_dihapus = int(mask_bukan_produk.sum())
df.drop(df.index[mask_bukan_produk], inplace=True)

print(f'Baris non-produk yang dihapus: {jumlah_dihapus}')

basket = pd.crosstab(
    index=df['InvoiceNo'],
    columns=df['Description']
).gt(0)

print('Ukuran basket:', basket.shape)
basket.head()

Rincian baris nonproduk yang akan dihapus:
Description
POSTAGE                                1126
DOTCOM POSTAGE                          708
Manual                                  328
CARRIAGE                                141
Next Day Carriage                        79
check                                    39
found                                    25
adjustment                               14
Bank Charges                             12
amazon                                    8
Found                                     8
Amazon                                    7
had been put aside                        5
dotcom                                    4
mailout                                   3
Adjust bad debt                           3
High Resolution Image                     3
AMAZON                                    3
AMAZON FEE                                2
Adjustment                                2
SAMPLES                                   2
returned             

C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\46279160.py:73: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(df.index[mask_bukan_produk], inplace=True)


Ukuran basket: (19797, 4022)


Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY,add stock to allocate online orders,alan hodge cant mamage this section
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536366,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536367,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536368,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536369,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [8]:
mask_sisa_nonproduk = (
    df['Description']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.casefold()
    .isin(deskripsi_bukan_produk)
    | df['StockCode'].astype(str).str.strip().str.upper().isin(kode_bukan_produk)
)
print('Kandidat nonproduk yang masih tersisa:', int(mask_sisa_nonproduk.sum()))
df.loc[mask_sisa_nonproduk, ['StockCode', 'Description']].head()

Kandidat nonproduk yang masih tersisa: 0


,StockCode,Description


In [22]:
from mlxtend.frequent_patterns import apriori, association_rules

min_support = 0.01
min_confidence = 0.6

frequent_itemsets = apriori(
    basket,
    min_support=min_support,
    use_colnames=True,
    low_memory=True
)

print('Jumlah frequent itemset:', len(frequent_itemsets))
frequent_itemsets.sort_values('support', ascending=False).head(10)

Jumlah frequent itemset: 1877


,support,itemsets
791,0.114159,frozenset({WHITE HANGING HEART T-LIGHT HOLDER})
357,0.105673,frozenset({JUMBO BAG RED RETROSPOT})
568,0.100470,frozenset({REGENCY CAKESTAND 3 TIER})
475,0.085164,frozenset({PARTY BUNTING})
397,0.079002,frozenset({LUNCH BAG RED RETROSPOT})
54,0.073496,frozenset({ASSORTED COLOUR BIRD ORNAMENT})
637,0.069960,frozenset({SET OF 3 CAKE TINS PANTRY DESIGN })
454,0.066677,frozenset({PACK OF 72 RETROSPOT CAKE CASES})
389,0.064303,frozenset({LUNCH BAG BLACK SKULL.})
428,0.063090,frozenset({NATURAL SLATE HEART CHALKBOARD })


In [23]:
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=min_confidence
)

kolom_aturan = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
rules_sorted = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print('Jumlah aturan asosiasi:', len(rules_sorted))
print('10 aturan teratas berdasarkan lift:')
display(rules_sorted[kolom_aturan].head(10))

print('3 aturan terbaik berdasarkan lift:')
display(rules_sorted[kolom_aturan].head(3))

Jumlah aturan asosiasi: 503
10 aturan teratas berdasarkan lift:


,antecedents,consequents,support,confidence,lift
0,frozenset({HERB MARKER THYME}),"frozenset({HERB MARKER ROSEMARY, HERB MARKER P...",0.010254,0.856540,78.869414
1,"frozenset({HERB MARKER ROSEMARY, HERB MARKER P...",frozenset({HERB MARKER THYME}),0.010254,0.944186,78.869414
2,"frozenset({HERB MARKER PARSLEY, HERB MARKER TH...",frozenset({HERB MARKER ROSEMARY}),0.010254,0.953052,78.614847
3,frozenset({HERB MARKER ROSEMARY}),"frozenset({HERB MARKER PARSLEY, HERB MARKER TH...",0.010254,0.845833,78.614847
4,"frozenset({HERB MARKER BASIL, HERB MARKER THYME})",frozenset({HERB MARKER ROSEMARY}),0.010103,0.952381,78.559524
5,frozenset({HERB MARKER ROSEMARY}),"frozenset({HERB MARKER BASIL, HERB MARKER THYME})",0.010103,0.833333,78.559524
6,frozenset({HERB MARKER THYME}),"frozenset({HERB MARKER BASIL, HERB MARKER ROSE...",0.010103,0.843882,78.066958
7,"frozenset({HERB MARKER BASIL, HERB MARKER ROSE...",frozenset({HERB MARKER THYME}),0.010103,0.934579,78.066958
8,frozenset({HERB MARKER THYME}),frozenset({HERB MARKER ROSEMARY}),0.011163,0.932489,76.918724
9,frozenset({HERB MARKER ROSEMARY}),frozenset({HERB MARKER THYME}),0.011163,0.920833,76.918724


3 aturan terbaik berdasarkan lift:


,antecedents,consequents,support,confidence,lift
0,frozenset({HERB MARKER THYME}),"frozenset({HERB MARKER ROSEMARY, HERB MARKER P...",0.010254,0.856540,78.869414
1,"frozenset({HERB MARKER ROSEMARY, HERB MARKER P...",frozenset({HERB MARKER THYME}),0.010254,0.944186,78.869414
2,"frozenset({HERB MARKER PARSLEY, HERB MARKER TH...",frozenset({HERB MARKER ROSEMARY}),0.010254,0.953052,78.614847


Berdasarkan eksperimen minimum confidence sebesar 0,5, 0,6, dan 0,7, nilai 0,6 dipilih sebagai threshold sementara karena memberikan keseimbangan antara cakupan aturan dan tingkat kepercayaan asosiasi. 

Tiga aturan dengan lift tertinggi menunjukkan keterkaitan kuat antara pembelian HERB MARKER THYME, HERB MARKER ROSEMARY, dan HERB MARKER PARSLEY. Confidence ketiga aturan berkisar antara 85,65% hingga 95,31%, dengan lift sekitar 78,61–78,87.

Dalam konteks bisnis ritel, pola tersebut dapat dijadikan dasar untuk menguji strategi product bundling, cross-selling, dan rekomendasi produk pelengkap. 

In [24]:
# Menampilkan aturan yang tidak melibatkan produk HERB MARKER.
aturan_non_herb = rules_sorted.loc[
    ~rules_sorted['antecedents'].apply(
        lambda items: any('HERB MARKER' in item.upper() for item in items)
    )
    & ~rules_sorted['consequents'].apply(
        lambda items: any('HERB MARKER' in item.upper() for item in items)
    )
].copy()

print('Jumlah aturan tanpa HERB MARKER:', len(aturan_non_herb))
display(aturan_non_herb[kolom_aturan].head(10))

Jumlah aturan tanpa HERB MARKER: 471


,antecedents,consequents,support,confidence,lift
32,frozenset({REGENCY TEA PLATE PINK}),"frozenset({REGENCY TEA PLATE GREEN , REGENCY T...",0.012881,0.836066,51.885863
33,"frozenset({REGENCY TEA PLATE GREEN , REGENCY T...",frozenset({REGENCY TEA PLATE PINK}),0.012881,0.799373,51.885863
34,frozenset({REGENCY TEA PLATE GREEN }),"frozenset({REGENCY TEA PLATE PINK, REGENCY TEA...",0.012881,0.669291,49.256359
35,"frozenset({REGENCY TEA PLATE PINK, REGENCY TEA...",frozenset({REGENCY TEA PLATE GREEN }),0.012881,0.947955,49.256359
36,frozenset({REGENCY TEA PLATE GREEN }),frozenset({REGENCY TEA PLATE PINK}),0.014043,0.729659,47.360836
37,frozenset({REGENCY TEA PLATE PINK}),frozenset({REGENCY TEA PLATE GREEN }),0.014043,0.911475,47.360836
38,"frozenset({REGENCY TEA PLATE ROSES , GREEN REG...",frozenset({REGENCY TEA PLATE GREEN }),0.010153,0.897321,46.625387
39,frozenset({SET OF 3 WOODEN STOCKING DECORATION}),frozenset({SET OF 3 WOODEN TREE DECORATIONS}),0.011062,0.686520,46.385815
40,frozenset({SET OF 3 WOODEN TREE DECORATIONS}),frozenset({SET OF 3 WOODEN STOCKING DECORATION}),0.011062,0.747440,46.385815
41,frozenset({REGENCY MILK JUG PINK }),frozenset({REGENCY SUGAR BOWL GREEN}),0.012275,0.738602,45.410249
